# TiA 1 — Depth, signal propagation and trainability

**Big question:** What actually goes wrong as a neural network becomes deep?

By the end, you should be able to connect products of Jacobians to vanishing/exploding signals, predict a useful initialisation scale, and explain what a residual path changes. Change one variable at a time and write a claim supported by each plot.

## How to work through this activity

This is a guided investigation rather than a coding tutorial. For each experiment:

1. Read the mathematical claim and identify the quantity being measured.
2. Predict the qualitative result before running the code.
3. Run one cell at a time and inspect both values and plots.
4. Change only the suggested variable; rerun and explain what changed.
5. Answer the **Explain** questions in your own words.

The code contains more comments than production software intentionally. You are not expected to memorise framework syntax. Focus on the relationship between assumptions, measurements and conclusions.

## Notation and prediction

For layer $l$, let $h_l\in\mathbb{R}^{n_l}$ be its activation, $W_l$ its weight matrix, $z_l=W_lh_l$ its pre-activation and $h_{l+1}=\phi(z_l)$. Backpropagation repeatedly applies

$$\frac{\partial L}{\partial h_l}=W_l^\top\operatorname{diag}\!\left(\phi'(z_l)\right)\frac{\partial L}{\partial h_{l+1}}.$$

Products of factors smaller or larger than one can therefore make gradients vanish or explode. Under independent, zero-mean assumptions, $\operatorname{Var}(z_l)\approx n_l\operatorname{Var}(W_l)\operatorname{Var}(h_l)$. Predict the weight variance that preserves scale for a linear activation and for ReLU.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Fix every random-number generator so that your plots match the reference run.
# After completing the guided activity, change the seed to test robustness.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# The default path is designed for a CPU. Set this to False only after the
# notebook works and you want to run longer variants.
FAST_MODE = True

## A. Watch signals propagate

For $h_{l+1}=\phi(W_lh_l)$, mean-field reasoning predicts that the variance is repeatedly multiplied by a factor involving fan-in, weight variance and the activation derivative. Compare arbitrary scaling with Xavier ($1/n$) and He ($2/n$).

In [ ]:
def propagate(depth=60, width=128, activation="relu", scale=1.0):
    h=rng.normal(size=(512,width)); variances=[]; saturated=[]
    for _ in range(depth):
        W=rng.normal(0,scale/np.sqrt(width),(width,width)); z=h@W
        h={"relu":lambda x:np.maximum(x,0),"tanh":np.tanh,"sigmoid":lambda x:1/(1+np.exp(-x))}[activation](z)
        variances.append(h.var()); saturated.append(np.mean(np.abs(h)>0.95) if activation=="tanh" else 0)
    return np.array(variances),np.array(saturated)

fig,ax=plt.subplots(1,2,figsize=(10,3))
for label,act,scale in [("small sigmoid","sigmoid",0.2),("large tanh","tanh",2.0),("ReLU: Xavier","relu",1.0),("ReLU: He","relu",np.sqrt(2))]:
    v,s=propagate(activation=act,scale=scale); ax[0].semilogy(v+1e-14,label=label)
ax[0].set(xlabel="layer",ylabel="activation variance"); ax[0].legend(fontsize=8)
for scale in [0.5,1.0,np.sqrt(2),2.0]:
    v,_=propagate(activation="relu",scale=scale); ax[1].semilogy(v+1e-14,label=f"scale={scale:.2f}")
ax[1].set(xlabel="layer",ylabel="variance"); ax[1].legend(fontsize=8); plt.show()

In [ ]:
# A numerical self-check complements the log-scale plot.
# Stable here means that the last-layer variance remains within one order of
# magnitude of the first; it is not a universal guarantee for trained nets.
he_variance,_=propagate(activation="relu",scale=np.sqrt(2))
small_variance,_=propagate(activation="relu",scale=.5)
large_variance,_=propagate(activation="relu",scale=2.)
print(f"He final/initial variance: {he_variance[-1]/he_variance[0]:.3f}")
print(f"Small-scale final/initial: {small_variance[-1]/small_variance[0]:.3e}")
print(f"Large-scale final/initial: {large_variance[-1]/large_variance[0]:.3e}")
assert .05 < he_variance[-1]/he_variance[0] < 20
assert small_variance[-1]/small_variance[0] < 1e-10
assert large_variance[-1]/large_variance[0] > 1e10

## B. Gradients and residual paths

Autograd is used only as a measuring instrument. The experiment compares a plain update with a residual update, $h_{l+1}=h_l+αF_l(h_l)$. It does **not** claim residual networks generalise better.

In [ ]:
import torch
def gradient_profile(depth=80,width=64,residual=False,alpha=0.1):
    # Keep references to intermediate activations so autograd exposes dL/dh_l.
    x=torch.randn(256,width,requires_grad=True); hs=[]; h=x
    for _ in range(depth):
        # tanh makes a centred residual branch; alpha controls its perturbation.
        W=torch.randn(width,width)/np.sqrt(width); z=torch.tanh(h@W)
        h=h+alpha*z if residual else z
        h.retain_grad(); hs.append(h)
    h.square().mean().backward()
    return np.array([q.detach().std().item() for q in hs]),np.array([q.grad.norm().item() for q in hs])
fig,ax=plt.subplots(1,2,figsize=(10,3))
profiles={}
for residual in [False,True]:
    a,g=gradient_profile(residual=residual); label="residual" if residual else "plain"
    profiles[label]=(a,g)
    ax[0].semilogy(a,label=label); ax[1].semilogy(g,label=label)
    print(f"{label:8s}: first/last activation std={a[0]:.3f}/{a[-1]:.3f}; minimum gradient norm={g.min():.3e}")
ax[0].set(title="activation scale",xlabel="depth"); ax[1].set(title="gradient norm",xlabel="depth")
for a in ax:a.legend();a.grid(alpha=.2)
plt.show()
assert profiles["residual"][0][-1] > 10*profiles["plain"][0][-1]
assert profiles["residual"][1].min() > 10*profiles["plain"][1].min()

### Explain

1. Which initialisation preserves variance for ReLU, and under what assumptions?
2. Why can stable forward activations coexist with unstable gradients?
3. What identity term appears in a residual block's Jacobian?

**Reading:** [Glorot & Bengio (2010)](https://proceedings.mlr.press/v9/glorot10a.html) and [He et al. (2016)](https://arxiv.org/abs/1512.03385).

## Expected pattern and limits

Small scales collapse, large scales explode, and He scaling keeps ReLU activation variance roughly stable. The residual experiment uses a small centred branch and should preserve both activation and gradient scale better than the plain deep map. This does not establish better generalisation, nor guarantee that every residual parameterisation is stable.